In [8]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter # uv add langchain_text_splitters
from langchain_core.vectorstores import InMemoryVectorStore

In [2]:
def to_document(doc, dept, status, text):
    return Document(
        page_content=text,
        metadata={
            "doc": doc,
            "dept": dept,
            "status": status,
        }
    )

CORPUS = [
    to_document("refund_current", "policy", "current", "Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we offer store credit for the unused months. Monthly subscriptions can be cancelled at any time, but the current month is not refunded."),
    to_document("faq_refund", "faq", "current", "FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it. Monthly plans are never refunded for the month already started, but you can cancel any time."),
    to_document("refund_2023", "policy", "archived", "ARCHIVED (2023) refund policy: annual subscriptions could be refunded within 30 days of purchase."),
    to_document("shipping", "policy", "current", "Shipping. Orders of 5 bags or more ship free. Orders placed before 12:00 UTC ship the same day. Standard delivery takes 3 to 5 working days in the UK and 7 to 10 working days elsewhere in Europe."),
    to_document("roasting", "ops", "current", "Roasting schedule. Light roasts are roasted every Tuesday and dark roasts every Friday at our Lisbon roastery. Beans are freshest for 30 days after the roast date when kept in a sealed bag."),
    to_document("blends", "product", "current", "The Drift blend is a light roast from Ethiopia with notes of jasmine and lemon. The Ember blend is a dark roast with notes of cocoa and orange peel."),
    to_document("support", "policy", "current", "Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours."),
    to_document("giftcards", "policy", "current", "Zorvia gift cards never expire and can be used on subscriptions and single bags."),
    to_document("wholesale", "sales", "current", "Wholesale accounts require a minimum order of 20 kg per month and get 15% off list price."),
    to_document("referral", "marketing", "current", "Referral programme: both the referrer and the friend get one free bag after the friend's first order."),
    to_document("equipment", "product", "current", "Equipment note: the GX-4471 hand grinder is out of stock until March. The GX-5000 electric grinder ships in 2 days."),
    to_document("brewing", "product", "current", "Brewing guide for the Drift blend. Use 15 g of coffee for 250 ml of water at 94 degrees C. Wet the grounds and wait 30 seconds, then pour slowly in circles. Total brew time should be about 3 minutes. If the cup tastes sour, grind finer; if it tastes bitter, grind coarser. Rinse the paper filter first to remove any papery taste."),
    to_document("company", "about", "current", "Zorvia Coffee Co. was founded in Lisbon in 2018 and ships compostable packaging."),
]

print(len(CORPUS), "documents")

13 documents


In [4]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

d:\projects\aiLearnings\learnings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2480.51it/s]


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [7]:
CURRENT = lambda d: d.metadata.get("status") == "current"

In [9]:
def build_all(docs=None):
    global store, chunks
    chunks = splitter.split_documents(docs or CORPUS)
    store = InMemoryVectorStore(embedding=embedding)
    store.add_documents(chunks)

In [13]:
build_all()
print(len(chunks[0].page_content))

229


In [14]:
def retrieve(question, k=3, filter=None):
    kwargs = {"filter": filter} if filter else {}
    return store.similarity_search_with_score(question, k=k, **kwargs)

def show(results):
    for doc, score in results:
        print(f"{score:.3f} [{doc.metadata['status']}] {doc.metadata['doc']}: {doc.page_content[:100]}....")

In [16]:
QUESTION = "What is the refund policy?"
show(retrieve(QUESTION, k=3, filter=CURRENT))

0.594 [current] faq_refund: FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it....
0.570 [current] refund_current: Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we off....
0.272 [current] referral: Referral programme: both the referrer and the friend get one free bag after the friend's first order....
